# 과제 · 진료기록 RAG 종합 실습 (구현 + 평가)
## JSON 자료에서 읽고, 출처와 함께 답하고, 평가까지

> **이번엔 직접 만듭니다.** 검색·컨텍스트 조립·출처 강제 프롬프트 — RAG의 핵심 3개를 **여러분이 채웁니다.**
> 그리고 "내 RAG가 잘 동작하는지"를 **평가**까지 해봅니다.
>
> **핵심 한 문장:** *"자유어는 의미검색(RAG), 조건은 정형 매칭 — 모든 주장에 출처를 달고, 그 결과를 평가로 확인한다."*

### 이 과제에서 여러분이 채울 것 (`# TODO`)
| # | 무엇 | 어디서 |
|---|------|--------|
| 1 | **검색 함수** `retrieve_guidelines` | STEP 4 |
| 2 | **컨텍스트 조립** `build_context` (출처 포함) | STEP 4 |
| 3 | **출처 강제 시스템 프롬프트** | STEP 5 |
| 4 | **평가 해석** + **생성 평가 조사** | STEP 6 |

> 제가 미리 준 것: JSON 로딩(STEP 1), FAISS 인덱스(STEP 2), 조건매칭(STEP 3), 평가 예제 `hit@k`(STEP 6).
> 비용 드는 호출은 **임베딩 1회 + 답변 생성 몇 회**뿐이고, 모델은 `gpt-4o-mini`로 고정돼 있습니다.

---
## STEP 0 · 환경 준비
`.env`에 `OPENAI_API_KEY`가 있으면 됩니다. (STEP 1·3·4·6 검색평가는 키 없이도 진행)

In [1]:
import os, json
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(".env", usecwd=True))
HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY:", "있음 ✅" if HAS_KEY else "없음 (임베딩·생성은 건너뜀)")
DATA_DIR = next((p for p in ["HW/NLP/day6", ".", "data"]
                 if os.path.exists(os.path.join(p, "guidelines.json"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("guidelines.json / drugs.json / specialists.json 위치를 찾지 못했습니다")
print("DATA_DIR:", DATA_DIR)
MODEL = "gpt-4o-mini"   # 비용 절약: 고정

OPENAI_API_KEY: 있음 ✅
DATA_DIR: .


---
## STEP 1 · JSON 자료 읽기 — (제공, 무료)
세 JSON을 읽습니다. 자료가 파일로 분리돼 있어, 자료를 고쳐도 코드는 안 바뀝니다.

In [2]:
def load_json(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as f:
        return json.load(f)

guidelines  = load_json("guidelines.json")
drugs       = load_json("drugs.json")
specialists = load_json("specialists.json")
print(f"가이드라인 {len(guidelines)} / 약물행 {len(drugs)} / 전문의 {len(specialists)}")

가이드라인 5 / 약물행 10 / 전문의 4


---
## STEP 2 · 가이드라인 임베딩 → FAISS — (제공, 임베딩 1회 유료)
가이드라인을 임베딩해 의미검색 인덱스를 만듭니다. 저장해두면 재실행은 0원.
각 문서에 `id`·`title`을 **metadata**로 붙여 둡니다 — **출처 표기**의 재료입니다.

In [3]:
INDEX_DIR = "faiss_guidelines"
vectordb = None
if HAS_KEY:
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document
    emb = OpenAIEmbeddings(model="text-embedding-3-small")
    if os.path.isdir(INDEX_DIR):
        vectordb = FAISS.load_local(INDEX_DIR, emb, allow_dangerous_deserialization=True)
        print("기존 인덱스 로드 (비용 0)")
    else:
        docs = [Document(page_content=g["title"] + "\n" + g["body"],
                         metadata={"id": g["id"], "title": g["title"], "evidence": g["evidence"]})
                for g in guidelines]
        vectordb = FAISS.from_documents(docs, emb)
        vectordb.save_local(INDEX_DIR)
        print("새 인덱스 생성 + 저장")
else:
    print("키 없음 — STEP 2 건너뜀")

/tmp/ipykernel_9687/2385908001.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


새 인덱스 생성 + 저장


---
## STEP 3 · 조건 매칭 도구 — (제공, 무료)
약물·전문의는 의미검색이 아니라 **정형 조건**으로 거릅니다. "eGFR 32"는 비슷한 걸 찾는 게 아니라 정확히 한 행을 집어야 하니까요.

In [4]:
def lookup_drug(drug_name, egfr):
    for r in drugs:
        if r["drug"] == drug_name and r["egfr_min"] <= egfr <= r["egfr_max"]:
            return r
    return {"error": f"{drug_name}의 eGFR {egfr} 구간 정보 없음"}

def find_specialist(specialty):
    hits = [s for s in specialists if s["specialty"] == specialty and s["referral_available"]]
    return hits[0] if hits else {"error": f"{specialty} 협진 가능 전문의 없음"}

print(lookup_drug("메트포르민", 32))
print(find_specialist("신장내과"))

{'id': 'D-MET-2', 'drug': '메트포르민', 'egfr_range': 'eGFR 30-44', 'egfr_min': 30, 'egfr_max': 44, 'dose': '최대 1000mg으로 감량, 신규 시작 권장하지 않음', 'contraindication': '탈수·조영제 검사 전후 일시 중단'}
{'id': 'S-001', 'name': 'K 교수', 'specialty': '신장내과', 'referral_available': True, 'note': '당뇨병성 만성콩팥병, eGFR 저하 동반 약물 조정 협진'}


---
## STEP 4 · 🔧 검색 + 컨텍스트 조립 — **여러분이 구현**

RAG의 심장입니다. 두 함수를 채우세요.

### TODO 1 — `retrieve_guidelines(query, k)`
자유어 쿼리로 가이드라인 **top-k**를 가져옵니다.
- 힌트: `vectordb.similarity_search(query, k=k)` 가 `Document` 리스트를 줍니다.
- 각 결과에서 `d.metadata["id"]`, `d.metadata["title"]`, `d.page_content` 를 꺼내 **dict 리스트**로 반환하세요. (출처 표기에 id·title이 필요합니다)

### TODO 2 — `build_context(...)`
가져온 자료들을 **출처를 붙여** 하나의 문자열로 묶습니다.
- 각 자료 앞에 `[가이드라인 id "title"]`, `[약물 ...]`, `[전문의 ...]` 같은 **출처 머리표**를 답니다.
- 이 머리표가 있어야 다음 STEP에서 모델이 출처를 인용할 수 있습니다.

In [5]:
def retrieve_guidelines(query, k=2):
    """자유어 쿼리로 가이드라인 top-k를 가져온다. 반환: dict 리스트(id,title,evidence,content)."""
    if vectordb is None:
        return []
    docs = vectordb.similarity_search(query, k=k)
    results = []
    for d in docs:
        results.append({
            "id": d.metadata.get("id"),
            "title": d.metadata.get("title"),
            "evidence": d.metadata.get("evidence"),
            "content": d.page_content,
        })
    return results


def build_context(guideline_hits, drug_hit=None, specialist_hit=None):
    """가져온 자료들을 '출처 머리표'와 함께 하나의 컨텍스트 문자열로 조립."""
    parts = []
    for g in guideline_hits:
        parts.append(
            f"[가이드라인 {g['id']} \"{g['title']}\"]\n"
            f"근거등급: {g.get('evidence', 'N/A')}\n{g['content']}"
        )
    if drug_hit and "error" not in drug_hit:
        parts.append(
            f"[약물 {drug_hit['id']} \"{drug_hit['drug']}\"]\n"
            f"적용 구간: {drug_hit['egfr_range']}\n"
            f"용량: {drug_hit['dose']}\n금기/주의: {drug_hit['contraindication']}"
        )
    if specialist_hit and "error" not in specialist_hit:
        parts.append(
            f"[전문의 {specialist_hit['id']} \"{specialist_hit['specialty']}\"]\n"
            f"담당: {specialist_hit['name']}\n협진 메모: {specialist_hit['note']}"
        )
    return "\n\n".join(parts)

#### ✅ 자가 점검 (키 있을 때만 검색 동작)

In [6]:
if vectordb is not None:
    hits = retrieve_guidelines("신기능 저하 당뇨 약물 조정", k=2)
    assert isinstance(hits, list) and len(hits) >= 1, "리스트를 반환해야 합니다"
    assert "id" in hits[0] and "content" in hits[0], "각 dict에 id·content 키가 필요합니다"
    ctx = build_context(hits, lookup_drug("메트포르민", 32), find_specialist("신장내과"))
    assert "가이드라인" in ctx and "약물" in ctx, "출처 머리표가 컨텍스트에 있어야 합니다"
    print("✅ STEP 4 자가 점검 통과")
    print(ctx[:200], "...")
else:
    print("키 없음 — STEP 4 점검 생략 (STEP 6 검색평가로 대체 확인 가능)")

✅ STEP 4 자가 점검 통과
[가이드라인 G-DM-003 "당뇨병 환자의 저혈당 예방과 관리"]
근거등급: II-B
당뇨병 환자의 저혈당 예방과 관리
저혈당은 고령, 신기능 저하, 인슐린 또는 설폰요소제 사용 환자에서 위험이 높다. 반복적 저혈당이 있거나 저혈당 무감지증이 있는 환자에서는 혈당 목표를 완화하고 저혈당 위험이 낮은 약제로 전환을 고려한다. 신기능이 저하된 환자에서 설폰요 ...


---
## STEP 5 · 🔧 출처 강제 답변 생성 — **여러분이 프롬프트 작성**

이제 컨텍스트를 모델에 주고 답하게 합니다. 핵심은 **시스템 프롬프트**예요.

### TODO 3 — `SYSTEM` 프롬프트
다음 규칙이 **반드시** 들어가야 합니다(이게 환각 방어의 핵심):
1. 제공된 참조 자료(컨텍스트)에 있는 내용만으로 답한다
2. 모든 의학적 주장 뒤에 `[출처: id, title]` 형식으로 근거를 단다
3. 컨텍스트에 없으면 **"제공된 자료에 없음"** 이라고 답한다 (지어내지 않는다)
4. 마지막에 보조 의견 면책 문구를 붙인다

In [16]:
# TODO 3: 위 4가지 규칙을 담은 시스템 프롬프트를 작성하세요
SYSTEM = """당신은 병원 참조 자료만 근거로 답하는 임상 보조 AI입니다.

규칙:
1. 답변은 반드시 사용자가 제공한 [참조 자료] 안의 정보만 사용합니다.
2. 모든 의학적 주장 또는 권고 문장 뒤에는 반드시 [출처: id, title] 형식의 근거를 붙입니다. 예: [출처: G-DM-002, 심혈관·신장 위험을 동반한 당뇨병의 약물 선택]
3. 참조 자료에 없는 내용은 추측하지 말고 '제공된 자료에 없음'이라고 답합니다.
4. 답변 마지막에는 '본 답변은 제공된 자료 기반의 보조 의견이며, 최종 진료 판단은 담당 의료진이 내려야 합니다.'를 그대로 붙입니다.

답변 작성 방식:
- 핵심 판단을 먼저 짧게 제시합니다.
- 이어서 근거를 문장 단위로 정리합니다.
- 출처 형식이 없는 문장은 쓰지 않습니다.
"""

import re

def _source_map_from_context(context):
    source_map = {}
    for line in context.splitlines():
        line = line.strip()
        m = re.match(r'^\[(가이드라인|약물|전문의)\s+([^\s]+)\s+\"([^\"]+)\"\]$', line)
        if m:
            _, source_id, title = m.groups()
            source_map[source_id] = title
    return source_map

def _normalize_citations(text, context):
    source_map = _source_map_from_context(context)

    def repl(match):
        raw = match.group(1).strip()
        if ',' in raw:
            return f'[출처: {raw}]'
        tokens = raw.split()
        source_id = tokens[-1] if tokens else raw
        title = source_map.get(source_id)
        if title:
            return f'[출처: {source_id}, {title}]'
        return match.group(0)

    text = re.sub(r'\[출처:\s*([^\]]+)\]', repl, text)
    lines = [line.rstrip() for line in text.splitlines()]
    trailing = []
    while lines and re.fullmatch(r'\[출처:\s*[^\]]+\]\s*', lines[-1].strip()):
        trailing.append(lines.pop().strip())
    trailing = list(reversed(trailing))
    if trailing:
        body = '\n'.join(lines).rstrip()
        cite_blob = ' '.join(trailing)
        if body:
            body_lines = body.splitlines()
            for i in range(len(body_lines) - 1, -1, -1):
                stripped = body_lines[i].strip()
                if stripped and '본 답변은 제공된 자료 기반의 보조 의견이며' not in stripped:
                    body_lines[i] = body_lines[i].rstrip() + ' ' + cite_blob
                    break
            text = '\n'.join(body_lines)
    return text

def answer(question, context):
    if not HAS_KEY:
        return "키 없음 — 생성 건너뜀"
    from openai import OpenAI
    client = OpenAI()
    msg = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"[참조 자료]\n{context}\n\n[질문]\n{question}"}],
    )
    return _normalize_citations(msg.choices[0].message.content, context)

# 엔드-투-엔드 1회 (생성 비용 발생)
if HAS_KEY:
    q = "65세 제2형 당뇨, eGFR 32. 메트포르민 계속 써도 되나? 콩팥 관리도 신경써야 할 듯."
    g = retrieve_guidelines("신기능 저하 당뇨 약물 조정", k=2)
    ctx = build_context(g, lookup_drug("메트포르민", 32), find_specialist("신장내과"))
    print(answer(q, ctx))
else:
    print("키 없음 — 데모 생략")

65세 제2형 당뇨 환자에서 eGFR 32의 경우, 메트포르민의 사용에 주의가 필요합니다.

메트포르민은 신기능 저하 환자에서 eGFR 수치에 따라 감량하거나 중단을 고려해야 하며, eGFR 30-44의 경우 최대 용량은 1000mg으로 감량해야 합니다. 신규 시작은 권장되지 않습니다. 따라서, 현재의 상황에서는 메트포르민 사용을 지속하기 전에 용량 조절이 필요할 수 있습니다. 또한, 당뇨병성 만성콩팥병과 관련된 신장 관리도 함께 고려해야 합니다 [출처: G-DM-001, 제2형 당뇨병 1차 약물 요법][출처: D-MET-2, 메트포르민].

본 답변은 제공된 자료 기반의 보조 의견이며, 최종 진료 판단은 담당 의료진이 내려야 합니다.


---
## STEP 6 · 📊 평가 — 내 RAG는 잘 동작하나?

만드는 것만큼 중요한 게 **평가**입니다. RAG 평가는 크게 두 축이에요:
- **검색 평가:** 질문에 맞는 *문서를 제대로 가져왔나?* (← 아래 예제 `hit@k`)
- **생성 평가:** 가져온 자료로 *충실하게(출처에 맞게) 답했나? 환각은 없나?* (← 여러분이 조사)

### (제공) 검색 평가 예제 — `hit@k`
"질문 ↔ 정답 가이드라인 id" 쌍을 미리 정해두고, **top-k 안에 정답이 들어왔는지** 비율을 잽니다.
LLM 없이 채점되니 비용 0입니다.

In [12]:
# 평가셋: 질문과 "이 가이드라인이 나와야 정답"인 id (정답 라벨은 사람이 만든다)
EVAL_SET = [
    {"q": "콩팥 안 좋은 당뇨 환자 약물 조정", "gold": "G-CKD-001"},
    {"q": "심혈관 위험 동반 당뇨 약 선택",   "gold": "G-DM-002"},
    {"q": "저혈당이 자꾸 오는 환자",         "gold": "G-DM-003"},
]

def hit_at_k(eval_set, k=2):
    """top-k 검색 결과 안에 정답 id가 있으면 hit. 평균 적중률 반환."""
    if vectordb is None:
        return None
    hits = 0
    for ex in eval_set:
        got_ids = [h["id"] for h in retrieve_guidelines(ex["q"], k=k)]
        ok = ex["gold"] in got_ids
        hits += int(ok)
        print(f"  [{'O' if ok else 'X'}] {ex['q']!r} → {got_ids} (정답 {ex['gold']})")
    score = hits / len(eval_set)
    print(f"hit@{k} = {score:.2f}")
    return score

if vectordb is not None:
    hit_at_k(EVAL_SET, k=2)
    hit_at_k(EVAL_SET, k=1)
else:
    print("키 없음 — 검색 평가는 키가 있어야 실행됩니다 (개념은 아래 해석/조사로)")

  [O] '콩팥 안 좋은 당뇨 환자 약물 조정' → ['G-CKD-001', 'G-DM-003'] (정답 G-CKD-001)
  [O] '심혈관 위험 동반 당뇨 약 선택' → ['G-DM-002', 'G-HTN-001'] (정답 G-DM-002)
  [O] '저혈당이 자꾸 오는 환자' → ['G-DM-003', 'G-DM-002'] (정답 G-DM-003)
hit@2 = 1.00
  [O] '콩팥 안 좋은 당뇨 환자 약물 조정' → ['G-CKD-001'] (정답 G-CKD-001)
  [O] '심혈관 위험 동반 당뇨 약 선택' → ['G-DM-002'] (정답 G-DM-002)
  [O] '저혈당이 자꾸 오는 환자' → ['G-DM-003'] (정답 G-DM-003)
hit@1 = 1.00


### TODO 4-a — 평가 결과 해석 (아래 마크다운 셀을 채우세요)
- `hit@2` 점수가 1.0이 아니라면, **어떤 질문에서 틀렸고 왜 그럴지** 한두 문장으로 적으세요.
- `k`를 1로 줄이면 점수가 어떻게 변할까요? 직접 바꿔 돌려보고 관찰을 적으세요.

### TODO 4-b — 🔎 생성 평가 조사 (코드 아님, 서술)
검색 평가(hit@k)는 *문서를 잘 골랐나*만 봅니다. 하지만 **답변이 그 문서에 충실한지**는 따로 평가해야 합니다.
> **"답변이 출처에 충실한가 / 환각은 없는가"를 평가하는 방법을 최소 2가지 찾아** 아래에 정리하세요.

각 방법마다 (1) **무엇을 재는지**, (2) **왜 필요한지**, (3) **우리 진료 데이터에 적용한다면 어떻게 할지**를 2~3문장으로.
(검색해서 찾아도 좋습니다. 키워드 힌트: faithfulness, groundedness, answer relevance, hallucination, LLM-as-judge)

**[TODO 4-a · 검색 평가 해석]**

- 틀린 질문과 이유: 이번 실행에서는 `hit@2 = 1.00`이어서 틀린 질문이 없었습니다. 세 평가 질문 모두 top-2 안에 정답 가이드라인 id가 포함되어, 현재 데이터 규모에서는 검색기가 의도한 문서를 안정적으로 찾아왔습니다.
- k=1로 바꿨을 때 관찰: 이번 실행에서는 `hit@1 = 1.00`이었습니다. 즉 세 질문 모두 정답 문서가 top-2 안에만 포함된 것이 아니라 실제 1순위로 검색되었고, 현재 평가셋에서는 검색 순위 품질도 충분히 안정적이었습니다.

---

**[TODO 4-b · 생성 평가 조사 — 최소 2가지]**

**① Faithfulness / Groundedness:**
- 무엇을 재나: 답변의 각 주장과 권고가 제공된 컨텍스트에 실제로 근거를 두고 있는지, 즉 출처와 답변 내용이 서로 일치하는지를 평가합니다.
- 왜 필요한가: RAG는 문서를 가져와도 모델이 그 위에 없는 내용을 덧붙여 말할 수 있으므로, 검색 성공과 별개로 환각 여부를 따로 봐야 합니다.
- 우리 진료 데이터 적용: 최종 답변의 문장별로 `[출처: id, title]`가 붙었는지 확인하고, 해당 문장이 `guidelines.json`, `drugs.json`, `specialists.json`의 원문과 모순되지 않는지 LLM-as-judge나 사람이 대조 검토할 수 있습니다.

**② Answer Relevance:**
- 무엇을 재나: 생성된 답변이 사용자의 질문에 직접적으로 답하고 있는지, 즉 질문의 핵심 요소를 빠뜨리지 않고 다루는지를 평가합니다.
- 왜 필요한가: 근거에 충실한 답변이라도 환자의 실제 질문인 `시작해도 되나`, `eGFR 구간은 어떤가`, `어느 과 협진이 필요한가`를 놓치면 실사용 가치가 떨어집니다.
- 우리 진료 데이터 적용: 미션 환자 질문을 기준으로 `약 시작 가능 여부`, `eGFR 조건`, `심부전 관련 근거`, `협진과` 네 항목이 답변에 모두 포함됐는지 체크리스트나 LLM 채점 프롬프트로 평가할 수 있습니다.


---
# 🎯 미션 — 이 환자를 끝까지 처리하라

이 과제의 목표는 하나입니다. 아래 **새 환자**를, 여러분이 만든 RAG 파이프라인으로 **검색 → 매칭 → 출처 있는 답변 → 평가**까지 끝내는 것.

> **환자:** *"58세 제2형 당뇨, eGFR 40, 심부전 동반. 다파글리플로진 시작해도 되나?"*

아래 단계가 곧 **미션을 푸는 순서**이자 **제출 항목**입니다. 위에서 아래로 따라가면 미션이 완성됩니다.

- **1단계 (STEP 4)** 검색·컨텍스트 구현 → STEP 4 자가점검 ✅ 통과
- **2단계 (STEP 5)** 출처 강제 프롬프트 작성
- **3단계 (아래 셀)** 미션 케이스에 적용 → 자가점검 `assert` 3개 통과 → (키 있으면) 답변 생성
- **4단계 (STEP 6)** `hit@k` 실행·해석(4-a) + 생성 평가 조사(4-b)

지금은 **3단계**입니다. 아래 셀에서 환자에 맞는 쿼리·약물·진료과를 넣어 자가점검을 통과시키세요.

In [17]:
patient = "58세 제2형 당뇨, eGFR 40, 심부전 동반. 다파글리플로진 시작해도 되나?"

# 적절한 쿼리/약물/진료과를 직접 정하세요
g_hits = retrieve_guidelines("심부전 동반 당뇨 환자 SGLT2 억제제 선택", k=2)
d_hit  = lookup_drug("다파글리플로진", 40)
s_hit  = find_specialist("순환기내과")

print("가이드라인:", [h["title"] for h in g_hits])
print("약물      :", d_hit)
print("전문의    :", s_hit)

# 자가 점검 (검색 함수가 구현돼 있어야 첫 줄이 통과)
assert g_hits and any("심혈관" in h["title"] or "위험" in h["title"] for h in g_hits), "가이드라인 검색을 다시"
assert d_hit.get("egfr_range") == "eGFR 25 이상", "약물/eGFR 매칭을 다시"
assert s_hit.get("specialty") == "순환기내과", "협진과를 다시"
print("\n✅ 미션 자가 점검 통과")

가이드라인: ['심혈관·신장 위험을 동반한 당뇨병의 약물 선택', '당뇨병성 만성콩팥병의 통합 관리']
약물      : {'id': 'D-DAPA-1', 'drug': '다파글리플로진', 'egfr_range': 'eGFR 25 이상', 'egfr_min': 25, 'egfr_max': 999, 'dose': '10mg 1일 1회', 'contraindication': '활동성 당뇨병성 케톤산증 금기'}
전문의    : {'id': 'S-002', 'name': 'L 교수', 'specialty': '순환기내과', 'referral_available': True, 'note': '심부전·죽상경화성 심혈관질환 동반 당뇨 협진'}

✅ 미션 자가 점검 통과


In [18]:
if HAS_KEY:
    ctx = build_context(g_hits, d_hit, s_hit)
    print(answer(patient, ctx))
else:
    print("키 없음 — 자가 점검 통과까지가 제출 핵심")

58세 제2형 당뇨 환자에서 eGFR 40 및 심부전이 동반된 경우, 다파글리플로진 시작이 가능합니다.

다파글리플로진은 eGFR 25 이상에서 사용 가능한 SGLT2 억제제로, 심부전이 동반된 경우에도 SGLT2 억제제가 입원 위험을 낮춘다고 권고됩니다. 따라서, 주어진 정보에 따르면 다파글리플로진을 시작하는 것이 적절할 것으로 보입니다. 다만, 사용 시 신기능을 정기적으로 모니터링하고 약물 용량 조정을 고려해야 합니다 [출처: G-DM-002, 심혈관·신장 위험을 동반한 당뇨병의 약물 선택; G-CKD-001, 당뇨병성 만성콩팥병의 통합 관리; D-DAPA-1, 다파글리플로진].

본 답변은 제공된 자료 기반의 보조 의견이며, 최종 진료 판단은 담당 의료진이 내려야 합니다.


### 🏁 다 됐는지 확인
위 미션의 **1~4단계**를 모두 마쳤다면 제출 준비 완료입니다. (각 단계의 자가점검 ✅ 통과 + STEP 6 서술 작성)

## 📌 요약
- **RAG는 직접 짜봐야 안다.** 검색 → 컨텍스트(출처 포함) → 출처 강제 생성, 이 3단이 뼈대.
- **출처 강제 = 환각 방어.** 모든 주장에 [출처]를 달게 하면 자료에 없는 말을 하기 어려워진다.
- **평가가 절반.** 검색 평가(hit@k)로 *문서를 잘 골랐나*, 생성 평가로 *충실히 답했나* — 두 축을 본다.